# Vision Transformer: An Image is Worth 16x16 Words

## Learning Objectives
1. Understand patch embeddings and how images become sequences
2. Implement Vision Transformer blocks with self-attention
3. Train ViT on image classification
4. Compare Vision Transformer with CNN (ResNet) approaches

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")

## Level 1: Patch Embedding

Core idea: Convert images to sequences of patch embeddings using linear projection

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert image to patch embeddings"""
    
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # Linear projection: Conv2d with stride=patch_size
        # This "patches" the image and projects to embed_dim
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)  # (B, embed_dim, H/patch_size, W/patch_size)
        x = x.flatten(2)  # (B, embed_dim, num_patches)
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

# Test patch embedding
patch_embed = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, embed_dim=768)
x = torch.randn(2, 3, 224, 224)
patches = patch_embed(x)

print(f"Input shape: {x.shape}")
print(f"Patch embeddings shape: {patches.shape}")
print(f"Number of patches: {patch_embed.num_patches}")
print(f"\nBreakdown:")
print(f"  Image: 224×224×3")
print(f"  Patch size: 16×16")
print(f"  Patches per image: 14×14 = 196")
print(f"  Each patch projected to: 768 dimensions")

## Level 2: Vision Transformer Block

Transformer block with self-attention and MLP, using LayerNorm instead of BatchNorm

In [ ]:
class TransformerBlock(nn.Module):
    """Vision Transformer block: Multi-head Attention + MLP"""
    
    def __init__(self, embed_dim=768, num_heads=12, mlp_dim=3072, dropout=0.1):
        super().__init__()
        # Layer normalization (NOT batch norm like in CNN)
        self.norm1 = nn.LayerNorm(embed_dim)
        
        # Multi-head self-attention
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        
        # Second layer norm
        self.norm2 = nn.LayerNorm(embed_dim)
        
        # MLP feed-forward network
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),  # GELU activation (smoother than ReLU)
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        # Self-attention with pre-normalization and residual
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out  # Residual connection
        
        # MLP with pre-normalization and residual
        x_norm = self.norm2(x)
        mlp_out = self.mlp(x_norm)
        x = x + mlp_out  # Residual connection
        
        return x

# Test transformer block
block = TransformerBlock(embed_dim=768, num_heads=12, mlp_dim=3072)
# Input: (batch_size, num_patches + cls_token, embed_dim)
x = torch.randn(2, 197, 768)  # 196 patches + 1 CLS token
output = block(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Block parameters: {sum(p.numel() for p in block.parameters()):,}")
print(f"\nKey differences from CNN:")
print(f"  - LayerNorm instead of BatchNorm")
print(f"  - Multi-head self-attention (global receptive field)")
print(f"  - GELU activation instead of ReLU")
print(f"  - Pre-normalization (norm before attention/MLP)")

## Real-World Example 1: Complete Vision Transformer on Synthetic Data

Full ViT model with CLS token, position embeddings, and transformer blocks

In [ ]:
class VisionTransformer(nn.Module):
    """Complete Vision Transformer for image classification"""
    
    def __init__(self, img_size=224, patch_size=16, num_classes=10, embed_dim=768,
                 num_heads=12, num_layers=12, mlp_dim=3072, dropout=0.1):
        super().__init__()
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        num_patches = self.patch_embed.num_patches
        
        # CLS token (learnable parameter, like in BERT)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Position embeddings (learnable 1D embeddings)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        nn.init.normal_(self.pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(dropout)
        
        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_dim, dropout)
            for _ in range(num_layers)
        ])
        
        # Layer norm before classification
        self.norm = nn.LayerNorm(embed_dim)
        
        # Classification head
        self.fc = nn.Linear(embed_dim, num_classes)
        
        # Initialize weights
        nn.init.normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        
        # Patch embedding: (B, H, W, C) → (B, num_patches, embed_dim)
        x = self.patch_embed(x)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, num_patches+1, embed_dim)
        
        # Add position embeddings
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # Apply transformer blocks
        for block in self.blocks:
            x = block(x)
        
        # Take CLS token for classification
        x = self.norm(x[:, 0])
        x = self.fc(x)
        
        return x

# Create ViT-Small for demonstration
model = VisionTransformer(
    img_size=224, patch_size=16, num_classes=10,
    embed_dim=384, num_heads=6, num_layers=12, mlp_dim=1536
).to(device)

# Create synthetic dataset
train_size, test_size = 500, 100
X_train = torch.randn(train_size, 3, 224, 224)
y_train = torch.randint(0, 10, (train_size,))
X_test = torch.randn(test_size, 3, 224, 224)
y_test = torch.randint(0, 10, (test_size,))

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

print(f"ViT-Small parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTraining ViT on synthetic data...")

# Training loop
train_losses = []
test_accs = []

for epoch in range(5):
    model.train()
    epoch_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    scheduler.step()
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    acc = 100 * correct / total
    train_losses.append(avg_loss)
    test_accs.append(acc)
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Test Acc={acc:.2f}%")

print("\nViT training complete!")

## Real-World Example 2: Patch Size Trade-off Analysis

Different patch sizes affect computation, accuracy, and detail preservation

In [ ]:
# Analyze impact of patch size
patch_sizes = [4, 8, 16, 32]
patch_analysis = []

for patch_size in patch_sizes:
    num_patches = (224 // patch_size) ** 2
    
    # Create model with this patch size
    vit_model = VisionTransformer(
        img_size=224, patch_size=patch_size, num_classes=10,
        embed_dim=384, num_heads=6, num_layers=12, mlp_dim=1536
    ).to(device)
    
    params = sum(p.numel() for p in vit_model.parameters())
    
    # Estimate FLOPs (rough)
    # Self-attention: O(N^2 * d) where N=num_patches, d=embed_dim
    flops_per_attn = num_patches ** 2 * 384 * 12  # 12 layers
    
    patch_analysis.append({
        'patch_size': patch_size,
        'num_patches': num_patches,
        'parameters': params,
        'relative_flops': flops_per_attn / 1e9  # Rough, in billions
    })

print(f"Patch Size Trade-off Analysis:")
print(f"\nPatch Size | Tokens | Parameters | Relative Compute")
print(f"-" * 55)
for info in patch_analysis:
    print(f"{info['patch_size']:2d}×{info['patch_size']:2d}      | {info['num_patches']:3d}    | {info['parameters']:,} | {info['relative_flops']:.2f}B")

print(f"\nKey observations:")
print(f"  - Smaller patches (4×4) → More tokens → Higher compute (O(N^2))")
print(f"  - Larger patches (32×32) → Fewer tokens → Lower compute, less detail")
print(f"  - Standard (16×16) → Good balance for 224×224 images")

## Real-World Example 3: Visualizing Attention Patterns

Understanding what patches the model attends to

In [ ]:
# Create a modified ViT that returns attention weights
class VisionTransformerWithAttention(VisionTransformer):
    """ViT that returns attention weights for visualization"""
    
    def forward_with_attention(self, x):
        B = x.shape[0]
        
        # Patch embedding
        x = self.patch_embed(x)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        # Add position embeddings
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # Store attention from last layer
        with torch.no_grad():
            for i, block in enumerate(self.blocks[:-1]):
                x = block(x)
            
            # Get attention from last block
            last_block = self.blocks[-1]
            x_norm = last_block.norm1(x)
            attn_out, attn_weights = last_block.attn(x_norm, x_norm, x_norm)
            x = x + attn_out
            
            x_norm = last_block.norm2(x)
            mlp_out = last_block.mlp(x_norm)
            x = x + mlp_out
        
        # Classification
        x = self.norm(x[:, 0])
        logits = self.fc(x)
        
        return logits, attn_weights

# Analyze what patches a random image attends to
vit_attn = VisionTransformerWithAttention(
    img_size=224, patch_size=16, num_classes=10,
    embed_dim=384, num_heads=6, num_layers=12, mlp_dim=1536
).to(device)
vit_attn.eval()

# Create a sample image
sample_image = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    logits, attn_weights = vit_attn.forward_with_attention(sample_image)

print(f"Attention weights shape: {attn_weights.shape}")
print(f"  (batch, num_heads, seq_len, seq_len)")
print(f"\nFor each image, the model computes {attn_weights.shape[1]} separate attention heads")
print(f"Each attention head learns different types of relationships between patches")
print(f"\nExample: Head 1 might focus on local details, Head 2 on global patterns, etc.")

## Key Takeaways

**Core idea:** Split images into patches, embed them as sequences, apply transformers

**Key components:**
| Component | Role | Key Detail |
|-----------|------|------------|
| Patch Embed | Convert image to tokens | Linear projection per patch |
| CLS Token | Aggregation mechanism | Learned, attends to all patches |
| Position Embed | Spatial information | Learnable 1D embeddings |
| Transformer Block | Compute representations | LayerNorm + multi-head attention + MLP |

**When to use ViT:**
- Have large pre-trained model available (ImageNet-21k, JFT-300M)
- Want global receptive field from layer 1
- Training on sufficient data (more than CNN needs)

**Related papers:**
- [ResNet](./01-resnet.md) - CNN alternative
- [Attention Is All You Need](../nlp/concepts/01-attention-is-all-you-need.md) - Original transformer
- [CLIP](../retrieval/concepts/02-clip.md) - ViT for multimodal tasks

In [ ]:
# Visualization: Training convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Loss convergence
axes[0].plot(train_losses, marker='o', linewidth=2, label='Training Loss')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=11)
axes[0].set_title('ViT-Small Convergence on Synthetic Data', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Test accuracy
axes[1].plot(test_accs, marker='s', linewidth=2, color='green', label='Test Accuracy')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1].set_title('ViT-Small Test Accuracy', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
axes[1].set_ylim([0, 100])

plt.tight_layout()
plt.savefig('/tmp/vit_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

print("Vision Transformer implementation complete!")
print(f"\nSummary:")
print(f"  - Patch embeddings convert images to sequences")
print(f"  - Transformers provide global receptive field from layer 1")
print(f"  - ViT scales better than CNNs with large datasets")
print(f"  - Patch size selection is critical for accuracy/efficiency trade-off")
print(f"  - Pre-training is essential (ViT needs JFT-300M or similar)")